# dARK 2.0 - Testing Notebook

This notebook demonstrates the dARK 2.0 system interaction, including Authority registration, NAAN authorization, and ARK lifecycle management.

## Prerequisites

1. **Start Besu:** `docker-compose up -d`
2. **Deploy Contracts:** `python3 deploy.py`
3. **Install Deps:** `pip install web3`

## 1. Setup & Connection

In [ ]:
import json
import configparser
from web3 import Web3

# Connect to Local Blockchain
RPC_URL = "http://localhost:8545"
w3 = Web3(Web3.HTTPProvider(RPC_URL))

# Deployer Account (Besu Development Account)
PRIVATE_KEY = "0xae6ae8e5ccbfb04590405997ee2d52d2b330726137b875053c36d94e974d162f"
account = w3.eth.account.from_key(PRIVATE_KEY)

print(f"✅ Connected: {w3.is_connected()}")
print(f"💰 Account: {account.address}")

## 2. Load Deployed Contracts

In [ ]:
# Read deployment info
config = configparser.ConfigParser()
config.read('deployed_contracts.ini')

auth_addr = config['Authority']['address']
auth_abi = json.loads(config['Authority']['abi'])
Authority = w3.eth.contract(address=auth_addr, abi=auth_abi)

dark_addr = config['dARK']['address']
dark_abi = json.loads(config['dARK']['abi'])
dARK = w3.eth.contract(address=dark_addr, abi=dark_abi)

print(f"🛡️ Authority at: {auth_addr}")
print(f"📦 dARK at: {dark_addr}")

## 3. Helper Function

In [ ]:
def send_tx(contract_func, gas=300000):
    tx = contract_func.build_transaction({
        'from': account.address,
        'nonce': w3.eth.get_transaction_count(account.address),
        'gas': gas,
        'gasPrice': w3.eth.gas_price
    })
    signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    
    if receipt['status'] == 0:
        raise Exception(f"Transaction Reverted! Gas Used: {receipt['gasUsed']}")
    
    return receipt

print("Helper ready.")

## 4. Authority Management
Check if our wallet is already registered, otherwise register a new Authority.

In [ ]:
UUID = "auth-uuid-notebook-01"

# Check if WALLET is already registered (since 1 wallet = 1 authority)
try:
    existing_uuid = Authority.functions.get_uuid_by_wallet(account.address).call()
    print(f"⚠️ Wallet already registered as Authority: '{existing_uuid}'")
    UUID = existing_uuid # Use the existing UUID
except:
    # Not registered, proceed to register
    print(f"Registering new Authority '{UUID}'...")
    receipt = send_tx(Authority.functions.register_authority(UUID, account.address))
    print(f"✅ Registered! Gas: {receipt['gasUsed']}")

## 5. NAAN Authorization
Authorize our specific NAAN to be managed by this Authority.

In [ ]:
NAAN = "55555"

if Authority.functions.is_authorized(account.address, NAAN).call():
    print(f"NAAN '{NAAN}' is already authorized.")
else:
    print(f"Authorizing NAAN '{NAAN}'...")
    receipt = send_tx(Authority.functions.authorize_naan(NAAN))
    print(f"✅ Authorized! Gas: {receipt['gasUsed']}")

## 6. ARK Creation
Create a new ARK using the authorized NAAN.

In [ ]:
NAME = "my-doc-v1"
URL = "https://example.com/doc/v1"
CID = "QmHash123"

if dARK.functions.ark_exists(NAAN, NAME).call():
    print(f"ARK '{NAAN}/{NAME}' already exists.")
else:
    print(f"Creating ARK '{NAAN}/{NAME}'...")
    # Note: Higher gas limit for creating ARK
    receipt = send_tx(dARK.functions.create_ark(NAAN, NAME, URL, CID), gas=500000)
    print(f"✅ Created! Gas: {receipt['gasUsed']}")

## 7. Resolution & Verification

In [ ]:
# Resolve URL
url = dARK.functions.resolve(NAAN, NAME).call()
print(f"🔍 Resolved URL: {url}")

# Get Full Data
data = dARK.functions.get_ark(NAAN, NAME).call()
print(f"📦 Full Data: {data}")